# Mortalidade por faixa etária
## Resultados Parciais da Análise Exploratória dos Dados 


In [ ]:
import json
import re
import unicodedata

import geobr
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display
from pathlib import Path

In [ ]:

MUNICIPIO = None
CODIGO_MUNICIPIO = None
REGIONAL_SAUDE = None

CSV_MORTALIDADE = "/home/anna/tcc/Tratamento/Mortalidade/tabela_mortalidade_faixa_etaria.csv"

ANOS_RANGE_MORTALIDADE = (1979, 2024)
ANOS_PASSO_MORTALIDADE = 1

ANOS_POR_BLOCO_ACUMULADO = 5

ANO_MALHA_MUNICIPIOS_MG = 2020

FAIXAS_ETARIAS_MORTALIDADE = [
    "0-9", "10-19", "20-29", "30-39",
    "40-49", "50-59", "60-69", "70 ou mais",
]

COR_BARRAS = "#6ca8e5"
COR_PONTOS = "#d73f3f"
LARGURA_GRAFICO = 1900
ALTURA_GRAFICO_MORTALIDADE = 1300  
X_MIN = 0
X_MAX = 13000

FONTE_TITULO = 22
FONTE_EIXO_TITULO = 16
FONTE_EIXO_TICK = 14
FONTE_LEGENDA = 14
FONTE_TEXTO_BARRA = 14
FONTE_TEXTO_PONTO = 12

TITULO_MORTALIDADE = "Mortalidade por faixa etária"
TITULO_MORTALIDADE_ACUM = "Mortalidade por faixa etária"

In [ ]:
def _norm_txt(txt: str) -> str:
    return (
        unicodedata.normalize("NFKD", str(txt))
        .encode("ascii", "ignore")
        .decode("ascii")
        .lower()
        .strip()
    )

def _norm_col(col: str) -> str:
    return re.sub(r"\s+", "_", _norm_txt(col))

def _find_first_col(df: pd.DataFrame, candidates_norm: set[str]) -> str | None:
    for col in df.columns:
        if _norm_col(col) in candidates_norm:
            return col
    return None


def _drop_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    drop_cols = [c for c in df.columns if _norm_col(c).startswith("unnamed")]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

def _fmt_int(n):
    try:
        return f"{int(n):,}".replace(",", ".")
    except Exception:
        return str(n)

def _sum_row_wise(frame: pd.DataFrame, cols: list[str]) -> pd.Series:
    if not cols:
        return pd.Series([0] * len(frame), index=frame.index, dtype="int64")
    return (
        frame[cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .sum(axis=1)
        .round()
        .astype(int)
    )

def _col_ano_faixa(ano: int, faixa: str) -> str:
    return f"{int(ano)}_{faixa}"


def _col_total_ano(ano: int) -> str:
    return f"Total_{int(ano)}"


def _acumular_em_blocos(por_ano: pd.Series, anos_por_bloco: int) -> pd.Series:
    serie = pd.to_numeric(por_ano, errors="coerce").fillna(0).astype(int).sort_index()
    anos = serie.index.astype(int).tolist()

    acumulado = {}
    bloco_atual = []

    for ano in anos:
        bloco_atual.append(ano)
        if len(bloco_atual) == int(anos_por_bloco):
            ano_final = bloco_atual[-1]
            acumulado[ano_final] = int(serie.loc[bloco_atual].sum())
            bloco_atual = []

    if bloco_atual:
        ano_final = bloco_atual[-1]
        acumulado[ano_final] = int(serie.loc[bloco_atual].sum())

    return pd.Series(acumulado).sort_index()


def aplicar_filtros(df: pd.DataFrame) -> pd.DataFrame:
 
    df = _drop_index_cols(df)
    out = df.copy()

    col_mun = _find_first_col(out, {"municipios", "municipio", "cidades", "cidade"})
    col_cod = _find_first_col(out, {"codigo", "cod"})
    col_reg = _find_first_col(out, {"regional_de_saude", "regional_saude", "regional"})

    if MUNICIPIO is not None:
        if not col_mun:
            raise KeyError("Não encontrei coluna de município para filtrar.")
        out = out[out[col_mun].astype(str).map(_norm_txt) == _norm_txt(MUNICIPIO)]

    if CODIGO_MUNICIPIO is not None:
        if not col_cod:
            raise KeyError("Não encontrei coluna de código para filtrar.")
        out = out[pd.to_numeric(out[col_cod], errors="coerce").astype("Int64") == int(CODIGO_MUNICIPIO)]

    if REGIONAL_SAUDE is not None:
        if not col_reg:
            raise KeyError("Não encontrei coluna de regional para filtrar.")
        out = out[out[col_reg].astype(str).map(_norm_txt) == _norm_txt(REGIONAL_SAUDE)]

    return out


print("\n===================================================\n")
print("MUNICIPIO:", MUNICIPIO)
print("CODIGO_MUNICIPIO:", CODIGO_MUNICIPIO)
print("REGIONAL_SAUDE:", REGIONAL_SAUDE)
print("ANOS_RANGE_MORTALIDADE:", ANOS_RANGE_MORTALIDADE)
print("ANOS_POR_BLOCO_ACUMULADO:", ANOS_POR_BLOCO_ACUMULADO)

df_mort = pd.read_csv(CSV_MORTALIDADE)
f_mort = aplicar_filtros(df_mort)
print(f"Linhas selecionadas (mortalidade): {len(f_mort)}")

ano_ini_m, ano_fim_m = map(int, ANOS_RANGE_MORTALIDADE)
anos_m = list(range(ano_ini_m, ano_fim_m + 1))

faixas_por_linha = pd.DataFrame(index=f_mort.index)
for faixa in FAIXAS_ETARIAS_MORTALIDADE:
    cols_faixa = [
        _col_ano_faixa(ano, faixa)
        for ano in anos_m
        if _col_ano_faixa(ano, faixa) in f_mort.columns
    ]
    faixas_por_linha[faixa] = _sum_row_wise(f_mort, cols_faixa)

por_faixa_mort = faixas_por_linha[FAIXAS_ETARIAS_MORTALIDADE].sum(axis=0).astype(int)

por_ano_mort = {}
for ano in anos_m:
    col_total = _col_total_ano(ano)
    if col_total in f_mort.columns:
        total_ano = pd.to_numeric(f_mort[col_total], errors="coerce").fillna(0).sum()
    else:
        cols_ano = [
            _col_ano_faixa(ano, faixa)
            for faixa in FAIXAS_ETARIAS_MORTALIDADE
            if _col_ano_faixa(ano, faixa) in f_mort.columns
        ]
        total_ano = _sum_row_wise(f_mort, cols_ano).sum()
    por_ano_mort[ano] = int(total_ano)

por_ano_mort_s = pd.Series(por_ano_mort).sort_index()
por_ano_mort_acum_s = _acumular_em_blocos(por_ano_mort_s, ANOS_POR_BLOCO_ACUMULADO)

print("Mortalidade — exemplo 40-49:", _fmt_int(int(por_faixa_mort.get("40-49", 0))))
if 2023 in por_ano_mort_s.index:
    print("Mortalidade — Total 2023:", _fmt_int(int(por_ano_mort_s.loc[2023])))
print("Blocos acumulados (últimos anos):", list(por_ano_mort_acum_s.index.astype(int)))

In [ ]:
def _xticks_fixos(x_min: int, x_max: int, passo: int = 500) -> list[int]:
    return list(range(int(x_min), int(x_max) + 1, int(passo)))

def plot_mortalidade(
    *,
    por_faixa: pd.Series,
    por_ano: pd.Series,
    titulo: str,
    anos_range: tuple[int, int],
    anos_passo: int = 1,
    yaxis2_title: str = "Ano",
    cor_barras: str = COR_BARRAS,
    cor_pontos: str = COR_PONTOS,
    width: int = LARGURA_GRAFICO,
    height: int = ALTURA_GRAFICO_MORTALIDADE,
    x_min: int = X_MIN,
    x_max: int = X_MAX,
):
    por_faixa = pd.to_numeric(por_faixa, errors="coerce").fillna(0)
    por_ano = pd.to_numeric(por_ano, errors="coerce").fillna(0)

    por_faixa = por_faixa[por_faixa > 0]
    por_ano = por_ano[por_ano > 0]
    if por_faixa.empty or por_ano.empty:
        raise ValueError("Sem dados > 0 para plotar.")

    categorias = por_faixa.index.astype(str).tolist()
    valores_barras = por_faixa.values.astype(int).tolist()

    anos_lista = por_ano.index.astype(int).tolist()
    valores_ano = por_ano.values.astype(int).tolist()

    ano_ini, ano_fim = map(int, anos_range)
    xticks = _xticks_fixos(x_min, x_max, passo=500)

    hover_barras = [
        f"<b>Faixa etária:</b> {cat}<br><b>Total:</b> {_fmt_int(val)}<br><i>Soma {ano_ini}–{ano_fim}</i>"
        for cat, val in zip(categorias, valores_barras)
    ]
    hover_pontos = [
        f"<b>{yaxis2_title}:</b> {ano}<br><b>Total:</b> {_fmt_int(val)}"
        for ano, val in zip(anos_lista, valores_ano)
    ]

    fig = go.Figure()

    fig.add_trace(go.Bar(
        name=f"Total por faixa etária (0 à 99+)",
        x=valores_barras,
        y=categorias,
        orientation="h",
        marker=dict(
            color=cor_barras,
            opacity=0.85,
            line=dict(color="white", width=1.0),
        ),
        text=[_fmt_int(v) for v in valores_barras],
        textposition="outside",
        textfont=dict(size=FONTE_TEXTO_BARRA, color="#222"),
        hovertext=hover_barras,
        hoverinfo="text",
        hovertemplate="%{hovertext}<extra></extra>",
        yaxis="y",
        xaxis="x",
    ))

    fig.add_trace(go.Scatter(
        name=f"Total por {yaxis2_title.lower()} ({ano_ini}–{ano_fim})",
        x=valores_ano,
        y=anos_lista,
        mode="markers+text",
        marker=dict(
            color=cor_pontos,
            size=11,
            opacity=0.92,
            line=dict(color="#d73f3f"),
            symbol="circle",
        ),
        text=[_fmt_int(v) for v in valores_ano],
        textposition="middle right",
        textfont=dict(size=FONTE_TEXTO_PONTO, color=cor_pontos),
        hovertext=hover_pontos,
        hoverinfo="text",
        hovertemplate="%{hovertext}<extra></extra>",
        yaxis="y2",
        xaxis="x",
    ))

    fig.update_layout(
        title=dict(
            text=titulo,
            font=dict(size=FONTE_TITULO, family="Arial", color="#1a1a2e"),
            x=0.5,
            xanchor="center",
            pad=dict(b=80),
        ),
        height=height,
        width=width,
        plot_bgcolor="#f8f9fb",
        paper_bgcolor="white",
        margin=dict(l=140, r=230, t=90, b=90),
        hovermode="closest",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.0,
            xanchor="center",
            x=0.5,
            font=dict(size=FONTE_LEGENDA),
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="#ddd",
            borderwidth=1,
        ),
        xaxis=dict(
            title="Número de mortes",
            tickvals=xticks,
            ticktext=[_fmt_int(v) for v in xticks],
            tickfont=dict(size=FONTE_EIXO_TICK),
            title_font=dict(size=FONTE_EIXO_TITULO),
            gridcolor="#dde3ed",
            gridwidth=1,
            showline=True,
            linecolor="#ccc",
            zeroline=True,
            zerolinecolor="#bbb",
            range=[x_min, x_max],
        ),
        yaxis=dict(
            title="Faixa etária",
            title_font=dict(size=FONTE_EIXO_TITULO),
            tickfont=dict(size=FONTE_EIXO_TICK),
            showgrid=False,
            showline=True,
            linecolor="#ccc",
            domain=[0, 1],
            type="category",
            categoryorder="array",
            categoryarray=categorias,
            showticklabels=True,
            automargin=True,
            tickmode="array",
            tickvals=categorias,
            ticktext=categorias,
        ),
        yaxis2=dict(
            title=yaxis2_title,
            title_font=dict(size=FONTE_EIXO_TITULO),
            tickfont=dict(size=FONTE_EIXO_TICK),
            overlaying="y",
            side="right",
            tickvals=anos_lista,
            showgrid=False,
            showline=True,
            linecolor="#ccc",
            range=[min(anos_lista) - 0.5, max(anos_lista) + 0.5],
        ),
        hoverlabel=dict(
            bgcolor="white",
            bordercolor="#aaa",
            font=dict(size=FONTE_EIXO_TICK, family="Arial"),
        ),
    )

    fig.update_layout(
        shapes=[
            dict(
                type="line",
                x0=0,
                x1=1,
                xref="paper",
                y0=cat,
                y1=cat,
                yref="y",
                line=dict(color="#dde3ed", width=1, dash="dot"),
            )
            for cat in categorias
        ]
    )

    fig.show()
    return fig


plot_mortalidade(
    por_faixa=por_faixa_mort.reindex(FAIXAS_ETARIAS_MORTALIDADE),
    por_ano=por_ano_mort_s,
    titulo=TITULO_MORTALIDADE,
    anos_range=ANOS_RANGE_MORTALIDADE,
    anos_passo=ANOS_PASSO_MORTALIDADE,
    yaxis2_title="Ano",
)

plot_mortalidade(
    por_faixa=por_faixa_mort.reindex(FAIXAS_ETARIAS_MORTALIDADE),
    por_ano=por_ano_mort_acum_s,
    titulo=f"{TITULO_MORTALIDADE_ACUM} (Acumulado {ANOS_POR_BLOCO_ACUMULADO} anos)",
    anos_range=ANOS_RANGE_MORTALIDADE,
    anos_passo=ANOS_POR_BLOCO_ACUMULADO,
    yaxis2_title="Fim do bloco",
)

print("Gráficos de mortalidade plotados (anual + acumulado)")

In [ ]:
if "f_mort" not in globals() or not isinstance(f_mort, pd.DataFrame):
    raise NameError("DataFrame 'f_mort' não encontrado. Execute a célula de preparação dos dados primeiro.")

base_num = f_mort.copy()

if "_find_first_col" in globals():
    col_mun_num = _find_first_col(base_num, {"municipios", "municipio", "cidades", "cidade", "nome_municipio"})
    col_cod_num = _find_first_col(base_num, {"codigo_municipio", "cod_municipio", "codigo", "cod", "codigo_ibge"})
else:
    col_mun_num, col_cod_num = None, None

if col_mun_num is None:
    candidatos = [c for c in base_num.columns if "mun" in str(c).lower() or "cidade" in str(c).lower()]
    col_mun_num = candidatos[0] if candidatos else None
if col_cod_num is None:
    candidatos = [c for c in base_num.columns if "cod" in str(c).lower()]
    col_cod_num = candidatos[0] if candidatos else None

if col_mun_num is None or col_cod_num is None:
    raise KeyError("Não foi possível identificar as colunas de município/código em f_mort.")

cols_total_ano = [c for c in base_num.columns if re.match(r"^Total_\d{4}$", str(c).strip())]

if cols_total_ano:
    anual = base_num[[col_mun_num] + cols_total_ano].copy()
    anual[cols_total_ano] = anual[cols_total_ano].apply(pd.to_numeric, errors="coerce").fillna(0)
    anual = anual.groupby(col_mun_num, as_index=True)[cols_total_ano].sum()
    anual.columns = [int(re.findall(r"\d{4}", c)[0]) for c in anual.columns]
    anual = anual.reindex(sorted(anual.columns), axis=1)
else:
    cols_ano_faixa = [c for c in base_num.columns if re.match(r"^\d{4}_.+", str(c).strip())]
    if not cols_ano_faixa:
        raise KeyError("Não encontrei colunas anuais (Total_YYYY ou YYYY_faixa) em f_mort.")

    anual_raw = base_num[[col_mun_num] + cols_ano_faixa].copy()
    anual_raw[cols_ano_faixa] = anual_raw[cols_ano_faixa].apply(pd.to_numeric, errors="coerce").fillna(0)
    anual_raw = anual_raw.groupby(col_mun_num, as_index=True)[cols_ano_faixa].sum()

    anos = sorted({int(str(c).split("_")[0]) for c in anual_raw.columns})
    anual = pd.DataFrame(index=anual_raw.index)
    for ano in anos:
        cols_ano = [c for c in anual_raw.columns if str(c).startswith(f"{ano}_")]
        anual[ano] = anual_raw[cols_ano].sum(axis=1)

anual = anual.sort_index(axis=1)

resumo = pd.DataFrame(index=anual.index)
resumo["Total de casos"] = anual.sum(axis=1)

top3_maior_total = resumo.nlargest(3, "Total de casos").reset_index()
if top3_maior_total.columns[0] != "Município":
    top3_maior_total = top3_maior_total.rename(columns={top3_maior_total.columns[0]: "Município"})

display(Markdown("### Top 3 — maior número de casos"))
display(top3_maior_total.style.format({"Total de casos": "{:,.0f}"}, decimal=",", thousands="."))


In [ ]:
def _peso_faixa_mort(df, filtro_faixa_etaria):
    
    peso_faixa = pd.DataFrame({
        "codigo_municipio": df["codigo_municipio"],
        "nome_municipio": df["nome_municipio"],
        "ano": df["ano"],
        "peso_faixa": 1.0
    })

    if filtro_faixa_etaria:
        peso_faixa = peso_faixa[
            peso_faixa["nome_municipio"].isin(filtro_faixa_etaria)
        ]

    return peso_faixa.reset_index(drop=True)


# Ensure the function is executed in the current scope
globals()["_peso_faixa_mort"] = _peso_faixa_mort

## Mapas de Minas Gerais por Município e Regional de Saúde

In [ ]:
ANO_INI_MAPA = 1979
ANO_FIM_MAPA = 2024
COR_SEM_MORTALIDADE = "#6ca8e5"
FAIXAS_MUNICIPIOS = [50, 100, 500, 1000, 5000]
PALETA_MAPA = ["#9c2328", "#d20000", "#ff7f50", "#ee9c21", "#fec44f", "#ffffd4"]

FAIXAS_REGIONAIS = [500, 1000, 1500, 2500, 5000]

In [ ]:


def _norm_cod7(s):
    x = pd.to_numeric(s, errors="coerce")
    return x.round().astype("Int64").astype(str).replace("<NA>", pd.NA).str.zfill(7)

def _first_col(df, candidatos):
    cols_norm = {str(c).strip().lower(): c for c in df.columns}
    for c in candidatos:
        if c.lower() in cols_norm:
            return cols_norm[c.lower()]
    return None

def _rotular_faixas_fixas(serie, faixas):
    s = pd.to_numeric(serie, errors="coerce").fillna(0)
    labels = ["Sem mortalidade (0)"]
    inicio = 1
    for i, limite in enumerate(faixas):
        if i == 0:
            labels.append(f"{inicio}–{limite}")
        else:
            labels.append(f"{faixas[i-1]+1:,}–{limite:,}".replace(",", "."))
        inicio = limite + 1
    labels.append(f"{faixas[-1]+1:,}+".replace(",", "."))
    classe = pd.Series("Sem mortalidade (0)", index=s.index)
    if (s > 0).any():
        inicio = 1
        idx_faixa = 1
        for limite in faixas:
            mask = (s >= inicio) & (s <= limite)
            classe.loc[mask] = labels[idx_faixa]
            inicio = limite + 1
            idx_faixa += 1
        classe.loc[s > faixas[-1]] = labels[-1]
    labels_usados = ["Sem mortalidade (0)"] + [lab for lab in labels[1:] if (classe == lab).any()]
    return classe, labels_usados

if "f_mort" in globals() and isinstance(f_mort, pd.DataFrame):
    base_mort = f_mort.copy()
elif "CSV_MORTALIDADE" in globals():
    raw_mort = pd.read_csv(CSV_MORTALIDADE)
    base_mort = aplicar_filtros(raw_mort) if "aplicar_filtros" in globals() else raw_mort
else:
    csv_mort_padrao = "/home/anna/tcc/Tratamento/Mortalidade/tabela_mortalidade_faixa_etaria.csv"
    raw_mort = pd.read_csv(csv_mort_padrao)
    base_mort = aplicar_filtros(raw_mort) if "aplicar_filtros" in globals() else raw_mort

if "_drop_index_cols" in globals():
    base_mort = _drop_index_cols(base_mort)

col_cod = _first_col(base_mort, {"Código","Codigo","codigo_municipio","cod_municipio","codigo","cod","codigo_ibge"})
col_mun = _first_col(base_mort, {"Municípios","Municipios","Município","Municipio","municipios","municipio","cidades","cidade","nome_municipio"})
col_reg = _first_col(base_mort, {"Regional de Saúde","Regional de Saude","Regional","regional_de_saude","regional_saude","regional"})

if col_cod is None or col_mun is None or col_reg is None:
    raise KeyError(f"Colunas não encontradas: cod={col_cod}, mun={col_mun}, reg={col_reg}\n{base_mort.columns.tolist()}")

anos = list(range(ANO_INI_MAPA, ANO_FIM_MAPA + 1))

cols_faixa = [
    c for c in base_mort.columns
    if re.match(r"^\d{4}_.+", str(c).strip())
    and int(str(c).split("_", 1)[0]) in set(anos)
]
if not cols_faixa:
    raise ValueError("Nenhuma coluna de faixa etária (YYYY_faixa) encontrada no DataFrame.")

acumulado_linha = base_mort[cols_faixa].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

base_acum = pd.DataFrame({
    "codigo_municipio": _norm_cod7(base_mort[col_cod]),
    "nome_municipio":   base_mort[col_mun].astype(str).str.strip(),
    "mortes_1979_2024": acumulado_linha,
})

por_municipio = (
    base_acum.groupby(["codigo_municipio","nome_municipio"], as_index=False)["mortes_1979_2024"].sum()
)

ano_malha = int(globals().get("ANO_MALHA_MUNICIPIOS_MG", 2020))
mg = geobr.read_municipality(code_muni="MG", year=ano_malha).to_crs(4326)
mg["codigo_municipio"] = _norm_cod7(mg["code_muni"])

mapa_mun = mg[["codigo_municipio","geometry"]].merge(por_municipio, on="codigo_municipio", how="left")
mapa_mun["nome_municipio"]   = mapa_mun["nome_municipio"].fillna("")
mapa_mun["mortes_1979_2024"] = pd.to_numeric(mapa_mun["mortes_1979_2024"], errors="coerce").fillna(0)

classe_mun, ordem_mun = _rotular_faixas_fixas(mapa_mun["mortes_1979_2024"], FAIXAS_MUNICIPIOS)
mapa_mun["classe_mortes"] = classe_mun

cores_mun = {"Sem mortalidade (0)": COR_SEM_MORTALIDADE}
labels_positivos_mun = [x for x in ordem_mun if x != "Sem mortalidade (0)"]
for i, lab in enumerate(reversed(labels_positivos_mun)):
    cores_mun[lab] = PALETA_MAPA[min(i, len(PALETA_MAPA)-1)]

geojson_mun = json.loads(mapa_mun[["codigo_municipio","geometry"]].to_json())

fig_mun = px.choropleth(
    mapa_mun,
    geojson=geojson_mun,
    locations="codigo_municipio",
    featureidkey="properties.codigo_municipio",
    color="classe_mortes",
    color_discrete_map=cores_mun,
    category_orders={"classe_mortes": ordem_mun},
    hover_name="nome_municipio",
    hover_data={"codigo_municipio": True, "mortes_1979_2024": ":,.0f", "classe_mortes": True},
    labels={"classe_mortes": "Mortalidade (1979–2024)", "mortes_1979_2024": "Total de mortes"},
    title="Mortalidade acumulada por município (1979–2024)",
)
fig_mun.update_geos(fitbounds="locations", visible=False,
                    showcountries=False, showcoastlines=False,
                    showland=False, showframe=False)
fig_mun.update_traces(marker_line_color="#2d3436", marker_line_width=0.8)
fig_mun.update_layout(
    template="plotly_white",
    width=1250, height=820,
    margin=dict(l=10, r=10, t=80, b=10),
    title=dict(x=0.5, xanchor="center", font=dict(size=20)),
    legend=dict(orientation="h", yanchor="top", y=0.99,
                xanchor="center", x=0.5, title=dict(text="")),
)
fig_mun.show()
print("✅ Mapa por município gerado")

In [ ]:

sem_match = mapa_mun[mapa_mun["mortes_1979_2024"] == 0]
print(sem_match[["codigo_municipio", "nome_municipio"]].to_string())

print("______________________________________________________________")

print("Municípios na malha geobr:", len(mg))
print("Municípios na tabela:", len(por_municipio))
print("Municípios sem match:", len(sem_match))
print("______________________________________________________________")


print("Exemplos de código na malha:", mg["codigo_municipio"].head(3).tolist())
print("Exemplos de código na tabela:", por_municipio["codigo_municipio"].head(3).tolist())

In [ ]:

# ── Helpers ───────────────────────────────────────────────────────────────────
def _norm_cod7(s):
    x = pd.to_numeric(s, errors="coerce")
    return x.round().astype("Int64").astype(str).replace("<NA>", pd.NA).str.zfill(7)

def _first_col(df, candidatos):
    cols_norm = {str(c).strip().lower(): c for c in df.columns}
    for c in candidatos:
        if c.lower() in cols_norm:
            return cols_norm[c.lower()]
    return None

def categoria_reg(n, faixas):
    labels = [
        f"0–{faixas[0]}",
        *[f"{faixas[i]+1:,}–{faixas[i+1]:,}".replace(",", ".") for i in range(len(faixas)-1)],
        f"{faixas[-1]+1:,}+".replace(",", "."),
    ]
    if n <= faixas[0]:
        return labels[0]
    for i in range(1, len(faixas)):
        if n <= faixas[i]:
            return labels[i]
    return labels[-1]

def centro(gdf):
    b = gdf.total_bounds
    return (b[1] + b[3]) / 2, (b[0] + b[2]) / 2

# ── Leitura da base de mortalidade ────────────────────────────────────────────
if "f_mort" in globals() and isinstance(f_mort, pd.DataFrame):
    base_mort = f_mort.copy()
elif "CSV_MORTALIDADE" in globals():
    raw_mort = pd.read_csv(CSV_MORTALIDADE)
    base_mort = aplicar_filtros(raw_mort) if "aplicar_filtros" in globals() else raw_mort
else:
    csv_mort_padrao = "/home/anna/tcc/Tratamento/Mortalidade/tabela_mortalidade_faixa_etaria.csv"
    raw_mort = pd.read_csv(csv_mort_padrao)
    base_mort = aplicar_filtros(raw_mort) if "aplicar_filtros" in globals() else raw_mort

if "_drop_index_cols" in globals():
    base_mort = _drop_index_cols(base_mort)

col_cod = _first_col(base_mort, {"Código","Codigo","codigo_municipio","cod_municipio","codigo","cod","codigo_ibge"})
if col_cod is None:
    raise KeyError(f"Coluna de código não encontrada.\n{base_mort.columns.tolist()}")

# ── Soma acumulada por município ──────────────────────────────────────────────
anos = set(range(ANO_INI_MAPA, ANO_FIM_MAPA + 1))
cols_faixa = [
    c for c in base_mort.columns
    if re.match(r"^\d{4}_.+", str(c).strip())
    and int(str(c).split("_", 1)[0]) in anos
]
if not cols_faixa:
    raise ValueError("Nenhuma coluna de faixa etária (YYYY_faixa) encontrada.")

acumulado_linha = base_mort[cols_faixa].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

base_acum = pd.DataFrame({
    "codigo_municipio": _norm_cod7(base_mort[col_cod]),
    "mortes":           acumulado_linha,
})

# ── Leitura da tabela de municípios → regionais ───────────────────────────────
candidatos_base = [Path.cwd().resolve()] + list(Path.cwd().resolve().parents)
path_municipios = None
for base in candidatos_base:
    pm = base / "Tratamento" / "Municípios" / "tabela_municipios_regionais.csv"
    if pm.exists():
        path_municipios = pm
        break

if path_municipios is None:
    raise FileNotFoundError("Não foi possível localizar tabela_municipios_regionais.csv")

df_mun_reg = pd.read_csv(path_municipios)
df_mun_reg.columns              = [c.strip() for c in df_mun_reg.columns]
df_mun_reg["codigo_ibge"]       = _norm_cod7(df_mun_reg["codigo_ibge"])
df_mun_reg["regional_original"] = df_mun_reg["regional_original"].fillna("Nenhum")

# ── Join mortalidade × regional → soma por regional ──────────────────────────
base_reg = base_acum.merge(
    df_mun_reg[["codigo_ibge", "regional_original"]],
    left_on="codigo_municipio",
    right_on="codigo_ibge",
    how="left",
)
base_reg["regional_original"] = base_reg["regional_original"].fillna("Nenhum")

por_regional = (
    base_reg[base_reg["regional_original"] != "Nenhum"]
    .groupby("regional_original", as_index=False)["mortes"].sum()
)

# ── Categorias e legenda ──────────────────────────────────────────────────────
ordem_reg = [
    f"0–{FAIXAS_REGIONAIS[0]}",
    *[f"{FAIXAS_REGIONAIS[i]+1:,}–{FAIXAS_REGIONAIS[i+1]:,}".replace(",", ".") for i in range(len(FAIXAS_REGIONAIS)-1)],
    f"{FAIXAS_REGIONAIS[-1]+1:,}+".replace(",", "."),
]
cores_reg = dict(zip(reversed(ordem_reg), PALETA_MAPA))

por_regional["categoria_mortes"] = por_regional["mortes"].apply(
    lambda n: categoria_reg(n, FAIXAS_REGIONAIS)
)

# ── Geometria: municípios → preenche sem regional → dissolve por regional ─────
ano_malha = int(globals().get("ANO_MALHA_MUNICIPIOS_MG", 2020))
mg = geobr.read_municipality(code_muni="MG", year=ano_malha).to_crs(4326)
mg["codigo_municipio"] = _norm_cod7(mg["code_muni"])

gdf_mg_reg = mg[["codigo_municipio", "geometry"]].merge(
    df_mun_reg[["codigo_ibge", "regional_original"]],
    left_on="codigo_municipio",
    right_on="codigo_ibge",
    how="left",
)
gdf_mg_reg["regional_original"] = gdf_mg_reg["regional_original"].fillna("Nenhum")

# Preenche municípios sem regional com o vizinho mais próximo
sem_regional = gdf_mg_reg["regional_original"] == "Nenhum"
if sem_regional.any():
    com_regional = gdf_mg_reg[~sem_regional].copy()
    for idx in gdf_mg_reg[sem_regional].index:
        geom = gdf_mg_reg.loc[idx, "geometry"]
        distancias = com_regional.geometry.distance(geom)
        gdf_mg_reg.loc[idx, "regional_original"] = com_regional.loc[distancias.idxmin(), "regional_original"]

gdf_regionais = (
    gdf_mg_reg[["regional_original", "geometry"]]
    .dissolve(by="regional_original", as_index=False)
)

# ── Merge geometria × mortalidade por regional ────────────────────────────────
gdf_regionais = gdf_regionais.merge(por_regional, on="regional_original", how="left")
gdf_regionais["mortes"]           = gdf_regionais["mortes"].fillna(0).astype(int)
gdf_regionais["categoria_mortes"] = gdf_regionais["categoria_mortes"].fillna(ordem_reg[0])
gdf_regionais["categoria_mortes"] = pd.Categorical(
    gdf_regionais["categoria_mortes"], categories=ordem_reg, ordered=True
)

gdf_regionais_plot       = gdf_regionais.reset_index(drop=True).copy()
gdf_regionais_plot["id"] = gdf_regionais_plot.index.astype(str)

# ── Mapa por regional ─────────────────────────────────────────────────────────
lat_c, lon_c = centro(gdf_regionais_plot)

fig_reg = px.choropleth_map(
    gdf_regionais_plot,
    geojson=json.loads(gdf_regionais_plot.to_json()),
    locations="id",
    color="categoria_mortes",
    color_discrete_map=cores_reg,
    category_orders={"categoria_mortes": ordem_reg},
    title=f"Mortalidade acumulada por Regional de Saúde ({ANO_INI_MAPA}–{ANO_FIM_MAPA})",
    hover_name="regional_original",
    hover_data={"mortes": True, "categoria_mortes": False, "id": False},
    labels={"mortes": "Total de mortes"},
    map_style="white-bg",
    center={"lat": lat_c, "lon": lon_c},
    zoom=5.5,
    opacity=0.95,
)
fig_reg.update_layout(
    margin=dict(l=10, r=10, t=60, b=10),
    title_x=0.5,
    height=760,
    legend_title_text="Mortes",
)
fig_reg.show()
print("✅ Mapa por regional gerado")

In [ ]:
# Ajustando o cálculo da média e desvio padrão de mortes por município
media_mortes_municipio = por_municipio['mortes_1979_2024'].mean()
desvio_padrao_mortes_municipio = por_municipio['mortes_1979_2024'].std()

print(f"Média de mortes por município (acumulado): {media_mortes_municipio:.2f}")
print(f"Desvio padrão de mortes por município (acumulado): {desvio_padrao_mortes_municipio:.2f}")

# Cálculo da média e desvio padrão anual
anos_totais = 2024 - 1979 + 1  # Inclui os anos de 1979 a 2024
total_mortes_anual = por_municipio['mortes_1979_2024'].sum() / anos_totais
desvio_padrao_anual = por_municipio['mortes_1979_2024'].sum() / anos_totais 

print(f"Média anual de mortes (total): {total_mortes_anual:.2f}")
print(f"Desvio padrão anual de mortes (total): {desvio_padrao_anual:.2f}")

In [ ]:

faixas_etarias = ['0–9', '10–19', '20–29', '30–39', '40–49', '50–59', '60–69', '70 ou mais']
valores_absolutos = [1, 11, 325, 2977, 7385, 10214, 9345, 12164]
percentuais = [0.002, 0.03, 0.77, 7.02, 17.41, 24.08, 22.03, 28.67]

tabela_faixa_etaria_acumulado = pd.DataFrame({
    'Faixa Etária': faixas_etarias,
    'Valor Absoluto': valores_absolutos,
    'Percentual (%)': percentuais
})

total_valor_absoluto = sum(valores_absolutos)
tabela_faixa_etaria_acumulado = pd.concat([
    tabela_faixa_etaria_acumulado,
    pd.DataFrame({
        'Faixa Etária': ['Total'],
        'Valor Absoluto': [total_valor_absoluto],
        'Percentual (%)': [100.0]
    })
], ignore_index=True)


display(tabela_faixa_etaria_acumulado)